<div>
<center><img src="../assets/Flux-logo.svg" width="400"/>
</div>

# Chapter 4: Flux with User-space Kubernetes

We are developing a new setup where it is possible to run Flux alongside user-space Kubernetes, or Usernetes.
This is a small demo of what that setup affords. In this tutorial, we will:
1. Start Usernetes
2. Run LAMMPS with Flux
3. Run an AI/ML training Job 
4. Run MuMMI Components with the Flux Operator (Flux -> Kubernetes -> Flux)
5. Run simulations alongside an AI/ML Model Server
<br>

Conceptually, this means that you can run traditional workloads using your HPC workload manager, and then use Kubernetes for associated services for AI/ML models, databases, and message queues.

![img/usernetes-flux.png](img/usernetes-flux.png)

To get started, double click on <span style="color:green; font-weight:600">usernetes-workspace.jupyterlab-workspace</span> in the file browser to the left. Double click until the page reloads and you have a terminal alongside this notebook.

## 1. Start Usernetes

Normally, we start Usernetes under a Flux batch job, meaning you (the user) do not have to do it, and do not see it. Here, we are going to show you all the setup. Run the following script <strong>in the terminal to your left</strong> to watch your control plane come up!

```bash
bash /home/ubuntu/start-usernetes.sh
```

When the script finishes, you'd normally export the `KUBECONFIG` to your environment. This is your credentials to interact with the cluster. However, you don't need to because we write the default to `~/.kube/config`.

```bash
export KUBECONFIG=/home/ubuntu/usernetes/kubeconfig
```

Try looking at the nodes:

```bash
kubectl get nodes
```

Right now you just have a single control plane, and we have modified it to allow for running work. This is how we map Usernetes nodes to physical nodes in HPC, with a 1:1 ratio. Finally, you can enable auto-complete (with TAB) for `kubectl`:

```bash
source <((kubectl completion bash))
```

Take a look at all the service pods! In Kubernetes we have the concept of [namespaces](https://kubernetes.io/docs/concepts/overview/working-with-objects/namespaces/) to organize things. By default we interact with `default`.

```bash
kubectl get pods --all-namespaces 
```

### Cache Containers

This is a strategy for pre-pulling (and caching) containers from your local filesystem. You might want to do this, for example, if you build an image locally and want to load it into the cluster without using a registry. Ensure you are in the chapter 4 tutorial directory, and then run the script for the following containers:

```bash
cd /home/ubuntu/tutorial/ch4
bash ./usernetes-load.sh ghcr.io/converged-computing/sc25-flux-eks:mlrunner-arm
bash ./usernetes-load.sh ghcr.io/converged-computing/sc25-flux-eks:createsims-arm
bash ./usernetes-load.sh ghcr.io/converged-computing/sc25-flux-eks:cganalysis-arm
bash ./usernetes-load.sh ghcr.io/converged-computing/flux-view-rocky:arm-9
bash ./usernetes-load.sh ghcr.io/converged-computing/lammps-reax-efa:ubuntu2404-efa
```

## 2. Run LAMMPS with Flux

<div class="alert alert-block" style="background-color:skyblue">
<span style="font-weight:600">Description:</span> Running LAMMPS with Flux in two ways 🍳 -- directly on the virtual machine, analogous to "bare metal" and then in user-space Kubernetes.
</div>

Let's start with a simple LAMMPS run, specifically LAMMPS with Flux Framework. This would be akin to sitting in an allocation you've created and interacting with your workload manager. 

```bash
flux run -N1 -n 8 --cwd /opt/lammps/examples/reaxff/HNS lmp -v x 2 -v y 2 -v z 2 -in in.reaxff.hns -nocite
```

Just as we learned in the previous tutorial, if you change `flux run` to `flux submit` the job will be non-blocking. You can also change the LAMMPS problem size to be larger to have a longer running time. 

### Flux in Kubernetes
Next, we are going to make a turducken - running Flux in Kubernetes, which (in our HPC setups) is already running under Flux! 
If we had more than one physical node, this would give us a powerful means to run HPC workloads in Kubernetes, with features that HPC cannot easily support such as elasticity and dynamism, declarative management, and modularity. 

<table>
    <tr>
<td style="width: 250px;"><img src="img/flux-usernetes-turkducken.png"></td>
        <td>Flux Framework and Usernetes setup. Your virtual machine is provisioned with both the workload manager Flux
Framework and User-space Kubernetes (1). We are emulating in this notebook you, the HPC user, running a batch job with Flux that has brought up your own user-space Kubernetes cluster (2). We will then deploy the Flux Operator (3) inside that cluster to run LAMMPS.</td>
    </tr>
</table>

Let's install the Flux Operator. Note we are installing for ARM.

```bash
kubectl apply -f https://raw.githubusercontent.com/flux-framework/flux-operator/refs/heads/main/examples/dist/flux-operator-arm.yaml
```

If you haven't, make sure you CD to the chapter 4 tutorial directory.

```bash
cd tutorial/ch4/
```

Create the Flux MiniCluster, which is the Custom Resource Definition (CRD) that the Flux Operator manages.

```bash
kubectl apply -f flux-minicluster-lammps.yaml
```

You can use `kubectl get pods` to watch the pods transition from `Init:0/1` (this is where we add Flux to the application container and configure the cluster) to `PodInitializing` (this is when your application container is being pulled) to `Running`. This entire sequence for this image takes approximately 3 minutes. Try using `--watch` to easily monitor for updates.

```bash
kubectl get pods --watch
```
When you see `Running` you can press Control+C.

Here is a way to use `kubectl get pods` with jq to programmatically get the lead broker pod identifier, which we can use to stream logs and watch LAMMPS output.

```bash
lead_broker=$(kubectl get pods -o json | jq -r .items[0].metadata.name)
kubectl logs $lead_broker -f
```

And when you are done, clean up the MiniCluster.

```bash
kubectl delete -f ./flux-minicluster-lammps.yaml
```

## 3. Run an AI/ML Training Job

<div class="alert alert-block" style="background-color:skyblue">
<span style="font-weight:600">Description:</span> Running AI/ML training jobs in Kubernetes is a first class citizen.
</div>

The newly released [Kubeflow Trainer](https://www.kubeflow.org/docs/components/trainer/getting-started/) project makes it easy to run Kubernetes components, specifically for Artificial Intelligence and Machine Learning workloads (AI/ML) in Kubernetes directly from Python. In your terminal to the left, use `kubectl` to install the Kubeflow Training Operator: 

```bash
# This is for JobSet and the Trainer Manager
VERSION=v2.0.0
kubectl apply --server-side -k "https://github.com/kubeflow/trainer.git/manifests/overlays/manager?ref=${VERSION}"

# This is for the runtimes (we need to give the webhook some time to create)
sleep 5
kubectl apply --server-side -k "https://github.com/kubeflow/trainer.git/manifests/overlays/runtimes?ref=${VERSION}"
```

If you get a `failed calling webhook` error, wait 30 seconds and try again. Hooks sometimes have a delay in starting up.
Next, use the kubeflow trainer Python sdk to see the available training runtimes. Do you like Python and dislike YAML? While we won't use it in this tutorial, the entire interaction of using Kubeflow in Kubernetes can be done using the [Python SDK](https://www.kubeflow.org/docs/components/trainer/getting-started/).

In [2]:
# Check available training runtimes
from kubeflow.trainer import TrainerClient, CustomTrainer

import os
os.putenv("KUBECONFIG", "/home/ubuntu/usernetes/kubeconfig")
for r in TrainerClient().list_runtimes():
    print(f"Runtime: {r.name}")

Runtime: deepspeed-distributed
Runtime: mlx-distributed
Runtime: mpi-distributed
Runtime: torch-distributed
Runtime: torchtune-llama3.2-1b
Runtime: torchtune-llama3.2-3b


While we can use the Python SDK for all our interactions, let's create the job the "old school" way - by applying a YAML file. Take a look at [pytorch-mnist.yaml](pytorch-mnist.yaml) and then in your terminal, run the job in your cluster by using `kubectl apply` with `-f` for a file.

```bash
kubectl apply -f ./pytorch-mnist.yaml
```

To see the pods, you can use `kubectl get` on the Pod resource type. Note that we will have two. Index 0 is the master, and 1 is the worker.

```bash
kubectl get pods

# More information about the hosts in "output wide" mode
kubectl get pods -o wide
```

And then get the pod identifier and look at the output. The `-f` will keep the output streaming.

```bash
kubectl logs pytorch-simple-node-0-0-xxxx -f
```

When the logs appear to be done, check the pods to see they are `Completed`

```bash
kubectl get pods
```

When you are ready to clean up, just `kubectl delete` the same file. Note that a pod in `Completed` state does not consume resources.

```bash
kubectl delete -f pytorch-mnist.yaml
```

Congratulations - you just ran your first AI/ML Job in User-space Kubernetes!

## 4. Run MuMMI Components with the Flux Operator

<div class="alert alert-block" style="background-color:skyblue">
<span style="font-weight:600">Description:</span> Orchestrate the MuMMI mlrunner with a an entire HPC cluster (1 node) in Kubernetes with the Flux Operator.
</div>

For this step, we are going to use the Flux Operator to run components of a well-known complex workflow, MuMMI. 


<table>
    <tr>
<td style="width: 600px;"><img src="./img/mummi.svg"></td>
        <td> MuMMI and its mini variant use a custom workload manager to orchestrate simulations leveraging Flux and backed by a message queue and the filesystem. Running in Kubernetes, we use an OCI registry to share assets between nodes, and a state machine design instead of a message queue.</td>
    </tr>
</table>

You can choose to run whichever component you like:

- `mlrunner`: This is a machine learning runner that generates simulation data to kick off a sequence of steps (pull time is longer, ~6 minutes due to model inside)
- `createsims`: Simulation setup that follows the machine learning step.
- `cganalysis`: Simulation (runs for 6 minutes with our configuration) but can take 12+ hours on HPC.

Choose and create  MuMMI component jobs using the examples below. 

```bash
# Run the mlrunner, the first step in MuMMI
kubectl apply -f flux-mummi-mlrunner.yaml

# Run the createsims step, the second step
kubectl apply -f flux-mummi-createsims.yaml

# Run the cganalysis step, the third step
kubectl apply -f flux-mummi-cganalysis.yaml
```

This container has a large model in it, and will be in `Init:0/1` and then `PodIniitalizing` to pull the container before it is `Running`. For any workflow component you choose, you can again look at the logs for the created pod. Note that the `mlrunner` will extract a model and run quickly (under 20 seconds), the `createsims` setup takes approximately 10 minutes, and `cganysis` is configured to run in ~6.

```bash
kubectl logs mlrunner-0-xxxx -f
kubectl logs createsims-0-xxxx -f
kubectl logs cganalysis-0-xxxx -f
```

## 5. Run Simulations alongside an AI/ML Model Server

<div class="alert alert-block" style="background-color:skyblue">
<span style="font-weight:600">Description:</span> Usernetes is powerful to provide a means to run AI/ML services alongside traditional HPC.
</div>

The most likely use case for user-space Kubernetes working with HPC is supporting services for your job simulations or workloads. As an example, you might want to deploy a database, message queue, or model server. For this example, we will deploy a [streaming ML model server](https://github.com/converged-computing/lammps-stream-ml/) backed by [RiverML](https://riverml.xyz) that allows you to build models that predict LAMMPS running time from input parameters X, Y, and Z. 

### Setup of Server

While there are many ways to expose service ports, we will use a simple means to enable ingress via Nginx.

```bash
kubectl apply -f https://raw.githubusercontent.com/kubernetes/ingress-nginx/main/deploy/static/provider/kind/deploy.yaml
kubectl wait --namespace ingress-nginx \
  --for=condition=ready pod \
  --selector=app.kubernetes.io/component=controller \
  --timeout=90s
```

The second command above ensures that the container is `Running` before we continue. Let's next deploy the ML server that will provide River models.

```bash
kubectl apply -f ml-server-deployment.yaml
```

Take a look at the service and ingress that are allowing for us to interact with the service, and when the ml-server-xxxx container is running, take a look at its logs to see the database being created and service starting.

```bash
kubectl get service ml-service-endpoint
kubectl get ingress ml-server-ingress
```

When you send a request to your node's public IP address on port 80, the flow is:

```console
Your Request -> Host Port 80 -> Ingress NGINX Pod -> ml-service-endpoint (Service) -> ml-server Pod (on port 8080)
```

There are a lot of layers! Importantly, when we run HPC applications like simulations inside of Usernetes, we use a host bypass mechanism (e.g., EFA or Infiniband) that will not be subject to the overhead added by [slirp4netns](https://arxiv.org/html/2402.00365v1).

### Test Server

Let's test it! The `/api` endpoint should deliver metadata about the service.

```bash
 curl -ks localhost/api/ | jq
```
```python
{
  "id": "django_river_ml",
  "status": "running",
  "name": "Django River ML Endpoint",
  "description": "This service provides an api for models",
  "documentationUrl": "https://vsoch.github.io/django-river-ml",
  "storage": "shelve",
  "river_version": "0.21.0",
  "version": "0.0.21"
}
```

Awesome! Let's next create our models. We are going to create [three regression](https://riverml.xyz/latest/api/linear-model/LinearRegression/) types. You can take a look at [model-scripts/1-create-models.py](model-scripts/1-create-models.py) to see how we will do that. River is akin to scikit-learn and other libraries that can create models from scratch or load on demand. We are creating them from scratch.

```bash
python3 model-scripts/1-create-models.py
```

We are next going to train our models directly on bare metal. We will submit jobs to Flux to run LAMMPS. Each will randomly select problem sizes, run, and send the features (X, Y, and Z problem size) and the value we want to predict, Y (the running time) to all the models. in our server. The default number of iterations is 20, but of course, for a better model you might consider increasing it.

```bash
python3 model-scripts/2-run-lammps-flux.py train --iters 100
```

Once your models are trained, let's create predictionst to test. This will save an output file to your local machine that we can generate plots with.

```bash
python3 model-scripts/2-run-lammps-flux.py predict --out test-predict.json
```

### Example Results

Here are metrics for models trained with 100 iterations.

```console
⭐️ Performance for: angry-truffle
          R Squared Error: 0.6087271917414379
       Mean Squared Error: 4.0731499339716315
      Mean Absolute Error: 1.6454122148608918
  Root Mean Squared Error: 2.0182046313423303

⭐️ Performance for: frigid-nalgas
          R Squared Error: -1.146171741238755
       Mean Squared Error: 22.34164782629544
      Mean Absolute Error: 4.571366798469615
  Root Mean Squared Error: 4.726695233066697

⭐️ Performance for: gassy-nunchucks
          R Squared Error: 0.8824891368445993
       Mean Squared Error: 1.2232880854477222
      Mean Absolute Error: 0.7817525756655499
  Root Mean Squared Error: 1.1060235465159511
```

<br>
<br>

## Debugging Tips 🪲

When something isn't working, here is a series of logical things to look at. First, the pod logs, or metadata in YAML.

```bash
# Pod logs, one shot
kubectl logs <pod>

# Pod logs, dangling
kubectl logs <pod> -f

# YAML manifest
kubectl get pod -o yaml
```

You might also want to use `describe` to see events.

```bash
kubectl describe pods <pod>
```

Often there is an associated service not working. Look for it, and then again use logs and describe to see what might be going on.

```bash
kubectl get pods --all-namespaces
```

When you use a `Deployment` or `Job` or `Service` it is often helpful to do the equivalent `get` or `describe` for that abstraction.